[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bernmor/mvp-previsao-demanda-bicicletas-seul/blob/main/seoul_bike_demand_mvp.ipynb)

# 0. Título e resumo

## Previsão de demanda horária de bicicletas compartilhadas em Seul

O seguinte notebook apresenta um projeto MVP para o curso de pós-graduação em Data Sciencce & Analytics. O intuito é prever a quantidade de bicicletas alugadas por hora no sistema de compartilhamento de bicicletas de Seul. Foi utilizado o dataset Seoul Bike sharing demand (originalmente encontrado em https:/archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand). O problema é tratado como uma tarefa supervisionada de regressão com estrutura temporal, usando informações de calendário, clima, estações do ano, feriados e a operação do serviço.

O fluxo presente neste notebook inclui uma análise exploratória, preparação dos dados, divisão temporal, análise de dimensionalidade, comparação de modelos, otimização de hiperparâmetros, avaliação final em dados não vistos, análise de erros, interpretabilidade, e discussão crítica.

A métrica escolhida para avaliação dos modelos será a MAE, pois ele representa diretamente o erro médio em número de bicicletas. RMSE e R² serão usados como métricas complementares para análise.

# 1. Configuração inicial

Esta seção concentra importações, constantes e pequenos ajustes globais. A intenção é deixar o notebook determinístico, simples de executar no Google Colab e fácil de modificar sem espalhar configurações pelo código.

In [ ]:
import math
import random
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
SEED = RANDOM_STATE
DATA_URL = "https://raw.githubusercontent.com/Bernmor/mvp-previsao-demanda-bicicletas-seul/main/data/SeoulBikeData.csv"
TARGET = "bicicletas_alugadas"
DATE_COLUMN = "data_hora"
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

COLUMN_RENAME = {
    "Date": "data",
    "Rented Bike Count": "bicicletas_alugadas",
    "Hour": "hora",
    "Temperature(°C)": "temperatura_c",
    "Humidity(%)": "umidade_pct",
    "Wind speed (m/s)": "velocidade_vento_ms",
    "Visibility (10m)": "visibilidade_10m",
    "Dew point temperature(°C)": "ponto_orvalho_c",
    "Solar Radiation (MJ/m2)": "radiacao_solar_mj_m2",
    "Rainfall(mm)": "chuva_mm",
    "Snowfall (cm)": "neve_cm",
    "Seasons": "estacao",
    "Holiday": "feriado",
    "Functioning Day": "dia_funcionamento",
}

SEASON_MAP = {
    "Winter": "inverno",
    "Spring": "primavera",
    "Summer": "verão",
    "Autumn": "outono",
}

HOLIDAY_MAP = {
    "Holiday": "feriado",
    "No Holiday": "não feriado",
}

FUNCTIONING_DAY_MAP = {
    "Yes": "sim",
    "No": "não",
}

PEAK_HOURS = {7, 8, 9, 17, 18, 19, 20}

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

print("Ambiente de execução:")
print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Configuração concluída com sucesso.")

# 2. Definição do problema
O contexto do projeto é analisar a demanda horária de um sistema de bicicletas compartilhadas. Em situações, isso é importante para estimar os períodos de maior procura, antecipar necessidade de reposicionamento de bicicletas e entender como o clima, calendário e funcionamento do serviço afetam o uso.

A variável alvo é `bicicletas_alugadas`, que representa a quantidade de bicicletas alugadas em uma determinada hora. Como o objetivo é prever um valor numérico contínuo ou de contagem, o problema será tratado como regressão supervisionada.

Neste caso a ordem temporal importa uma vez que para prever períodos futuros de demanda, é necessário treinar um modelo com dados de príodos passados. Portanto, a validação não deve embaralhar as observações, nem permitir que informações de períodos posteriores influenciem o treinamento ("data leakage"), a seleção de modelos ou a otimização de hiperparâmetros.

Uma premissa operacional importante é que variáveis como `hora`, `dia_semana`, `feriado`, `estacao` e `dia_funcionamento` seriam conhecidas antes da previsão. Em especial, `dia_funcionamento` é uma variável muito forte: se o serviço não opera, a demanda esperada cai drasticamente. Ela é mantida porque representa uma informação operacional planejada, não uma consequência observada depois do aluguel.

s principais restrições assumidas neste MVP são: uso apenas de dados públicos, previsão agregada por hora, ausência de variáveis externas como eventos locais ou disponibilidade por estação. O conjunto de teste será mantido isolado até a avaliação final.

# 3. Carregamento dos dados

Para esta entrega, os dados serão carregados diretamente de uma URL pública em formato raw do repositório GitHub.

In [ ]:
def load_seoul_bike_data(url: str = DATA_URL) -> pd.DataFrame:
    """Carrega o conjunto Seoul Bike Sharing Demand a partir de uma URL pública do GitHub."""
    return pd.read_csv(url, encoding="unicode_escape")

raw_df = load_seoul_bike_data()
print(f"Base carregada com {raw_df.shape[0]:,} linhas e {raw_df.shape[1]:,} colunas.")

# 4. Apresentação dos dados

A fonte dos dados é o conjunto público **Seoul Bike Sharing Demand**, disponibilizado pela UCI Machine Learning Repository. A base contém observações horárias do sistema de bicicletas compartilhadas de Seul. Cada linha representa uma hora, com a quantidade de bicicletas alugadas e variáveis explicativas relacionadas a calendário, clima, estação, feriado e funcionamento do serviço.

As principais variáveis incluem hora, temperatura, umidade, vento, visibilidade, ponto de orvalho, radiação solar, chuva, neve, estação do ano, feriado e indicação de funcionamento do serviço. A variável alvo é `bicicletas_alugadas`.

Limitações conhecidas: os dados são agregados por hora, não incluem demanda por estação de retirada, não trazem disponibilidade operacional em tempo real e não incorporam eventos locais, preço, greves ou alterações de política pública.

In [ ]:
summary_table = pd.DataFrame({
    "coluna": [COLUMN_RENAME.get(column, column) for column in raw_df.columns],
    "tipo": raw_df.dtypes.astype(str).values,
    "valores_ausentes": raw_df.isna().sum().values,
    "valores_unicos": raw_df.nunique(dropna=False).values,
})

summary_table

# 5. Análise exploratória inicial

Para a análise exploratória, busca-se entender a distribuição da demanda, padrões por hora e estação, diferenças em feriados e dias de funcionamento, efeitos climáticos e correlações entre variáveis numéricas.

## Padronização inicial para análise

Antes de produzir quaisquer gráficos exploratórios, nesta etapa vamos padronizar os nomes das colunas, a data e a hora devem ser combinadas em uma variável temporal única e algumas variáveis de calendário serão derivadas. Também traduzimos categorias do conjunto original para manter a análise em português.

Não serão criadas variáveis baseadas em valores futuros. Assim, esta preparação inicial preserva a ordem temporal e reduz o risco de vazamento de dados.

In [ ]:
df = raw_df.rename(columns=COLUMN_RENAME).copy()
df["data"] = pd.to_datetime(df["data"], dayfirst=True)
df[DATE_COLUMN] = df["data"] + pd.to_timedelta(df["hora"], unit="h")
df["estacao"] = df["estacao"].map(SEASON_MAP)
df["feriado"] = df["feriado"].map(HOLIDAY_MAP)
df["dia_funcionamento"] = df["dia_funcionamento"].map(FUNCTIONING_DAY_MAP)

df["mes"] = df[DATE_COLUMN].dt.month
df["dia_semana"] = df[DATE_COLUMN].dt.dayofweek
df["fim_de_semana"] = np.where(df["dia_semana"].isin([5, 6]), "sim", "não")
df["hora_seno"] = np.sin(2 * np.pi * df["hora"] / 24)
df["hora_cosseno"] = np.cos(2 * np.pi * df["hora"] / 24)
df["periodo_com_chuva"] = np.where(df["chuva_mm"] > 0, "sim", "não")
df["periodo_com_neve"] = np.where(df["neve_cm"] > 0, "sim", "não")
df["horario_pico"] = np.where(df["hora"].isin(PEAK_HOURS), "sim", "não")

df = df.sort_values(DATE_COLUMN).reset_index(drop=True)

if df.isna().any().any():
    raise ValueError("Foram encontrados valores ausentes após a preparação inicial.")

print(f"Período observado: {df[DATE_COLUMN].min()} até {df[DATE_COLUMN].max()}.")
df.head()

In [ ]:
display(df.head())

descriptive_statistics = df[[
    TARGET,
    "hora",
    "temperatura_c",
    "umidade_pct",
    "velocidade_vento_ms",
    "radiacao_solar_mj_m2",
    "chuva_mm",
    "neve_cm",
]].describe().T

missing_values = (
    df.isna()
    .sum()
    .rename("valores_ausentes")
    .reset_index()
    .rename(columns={"index": "coluna"})
)

print("Estatísticas descritivas das principais variáveis numéricas:")
display(descriptive_statistics)
print("Tabela de valores ausentes após a preparação inicial:")
display(missing_values)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

axes[0, 0].hist(df[TARGET], bins=40, color="#4C78A8", edgecolor="white")
axes[0, 0].set_title("Distribuição da demanda horária")
axes[0, 0].set_xlabel("Bicicletas alugadas por hora")
axes[0, 0].set_ylabel("Frequência")

demand_by_hour = df.groupby("hora", as_index=False)[TARGET].mean()
axes[0, 1].plot(demand_by_hour["hora"], demand_by_hour[TARGET], marker="o", color="#F58518")
axes[0, 1].set_title("Demanda média por hora do dia")
axes[0, 1].set_xlabel("Hora do dia")
axes[0, 1].set_ylabel("Média de bicicletas alugadas")
axes[0, 1].set_xticks(range(0, 24, 2))

demand_by_season = df.groupby("estacao", as_index=False)[TARGET].mean().sort_values(TARGET)
axes[0, 2].bar(demand_by_season["estacao"], demand_by_season[TARGET], color="#54A24B")
axes[0, 2].set_title("Demanda média por estação")
axes[0, 2].set_xlabel("Estação")
axes[0, 2].set_ylabel("Média de bicicletas alugadas")

demand_by_holiday = df.groupby("feriado", as_index=False)[TARGET].mean().sort_values(TARGET)
axes[1, 0].bar(demand_by_holiday["feriado"], demand_by_holiday[TARGET], color="#B279A2")
axes[1, 0].set_title("Demanda média por feriado")
axes[1, 0].set_xlabel("Feriado")
axes[1, 0].set_ylabel("Média de bicicletas alugadas")

demand_by_operation = df.groupby("dia_funcionamento", as_index=False)[TARGET].mean().sort_values(TARGET)
axes[1, 1].bar(demand_by_operation["dia_funcionamento"], demand_by_operation[TARGET], color="#E45756")
axes[1, 1].set_title("Demanda média por funcionamento do serviço")
axes[1, 1].set_xlabel("Dia de funcionamento")
axes[1, 1].set_ylabel("Média de bicicletas alugadas")

demand_by_peak = df.groupby("horario_pico", as_index=False)[TARGET].mean().sort_values(TARGET)
axes[1, 2].bar(demand_by_peak["horario_pico"], demand_by_peak[TARGET], color="#72B7B2")
axes[1, 2].set_title("Demanda média por horário de pico")
axes[1, 2].set_xlabel("Horário de pico")
axes[1, 2].set_ylabel("Média de bicicletas alugadas")

plt.tight_layout()
plt.show()

Os gráficos assima mostram que a distribuição da demanda é assimétrica: há muitas horas com demanda baixa ou moderada e menos horas com demanda muito alta. A média por hora sugere forte padrão diário, compatível com deslocamentos urbanos. A comparação por estação, feriado, funcionamento e horário de pico indica que variáveis de calendário e operação têm papel importante na demanda. Em sua maioria o perfil de aluguéis é em horários comerciais em dias de não feriado. O gráfico `Demanda média por horário de pico` mostra o padrão de aluguel coerente com de pessoas alugando bicicletas para ir e voltar do trabalho. Já o gráfico `Demanda média por estação` mostram que o alugel de bicicletas aumentam em estações de clima mais amenos ou com o aumento de turistas no verão, por exemplo. Para embasar estas especulações seria necessário cruzar os dados deste dataset com os dados presentes em outros tipos de dados (turistas em Seoul, por exemplo), mas a visualização dos dados do dataset mostram como diversos fatores podem influenciar o alguel de bicicletas e podem direcionar a análise posterior.

In [ ]:
weather_columns = ["temperatura_c", "chuva_mm", "neve_cm", "umidade_pct", "radiacao_solar_mj_m2"]
weather_labels = {
    "temperatura_c": "Temperatura (°C)",
    "chuva_mm": "Chuva (mm)",
    "neve_cm": "Neve (cm)",
    "umidade_pct": "Umidade (%)",
    "radiacao_solar_mj_m2": "Radiação solar (MJ/m²)",
}

fig, axes = plt.subplots(1, len(weather_columns), figsize=(20, 4))
for ax, column in zip(axes, weather_columns):
    ax.scatter(df[column], df[TARGET], alpha=0.25, s=10, color="#4C78A8")
    ax.set_title(f"Demanda versus {weather_labels[column]}")
    ax.set_xlabel(weather_labels[column])
    ax.set_ylabel("Bicicletas alugadas")

plt.tight_layout()
plt.show()

numeric_columns_for_correlation = [
    TARGET,
    "hora",
    "hora_seno",
    "hora_cosseno",
    "temperatura_c",
    "umidade_pct",
    "velocidade_vento_ms",
    "visibilidade_10m",
    "ponto_orvalho_c",
    "radiacao_solar_mj_m2",
    "chuva_mm",
    "neve_cm",
    "mes",
    "dia_semana",
]

correlation_matrix = df[numeric_columns_for_correlation].corr()
fig, ax = plt.subplots(figsize=(11, 9))
image = ax.imshow(correlation_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_title("Correlação entre variáveis numéricas")
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=90)
ax.set_yticklabels(correlation_matrix.columns)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

A temperatura e a radiação solar tendem a se relacionar com maior demanda em faixas intermediárias e altas, enquanto chuva e neve aparecem associadas a níveis menores de aluguel. A matriz de correlação ajuda a identificar relações lineares e possíveis redundâncias, mas não substitui modelos capazes de capturar interações e efeitos não lineares.

In [ ]:
eda_summary = pd.concat(
    [
        df.groupby("feriado")[TARGET].agg(["count", "mean", "median"]).rename_axis("grupo"),
        df.groupby("periodo_com_chuva")[TARGET].agg(["count", "mean", "median"]).rename_axis("grupo"),
    ],
    keys=["feriado", "periodo_com_chuva"],
)

eda_summary = eda_summary.rename(columns={"count": "quantidade", "mean": "media", "median": "mediana"})
eda_summary


A demanda média em feriados é menor do que em dias sem feriado, o que reforça a hipótese de uso associado a rotinas de deslocamento. Em períodos com chuva, a queda é ainda mais forte, indicando que condições climáticas adversas podem ser um dos principais fatores de redução da demanda.

# 6. Preparação dos dados

Para a modelagem, a coluna alvo será separada das variáveis explicativas. A coluna temporal será usada para ordenar os dados e fazer a divisão cronológica, mas não entrará diretamente como variável numérica contínua no modelo.

As decisões de engenharia de atributos são simples e evitam vazamento de dados:

- `mes` captura sazonalidade anual aproximada.
- `dia_semana` e `fim_de_semana` capturam diferenças de comportamento entre dias úteis e fins de semana.
- `hora_seno` e `hora_cosseno` representam a hora de forma cíclica, aproximando 23h e 0h em modelos lineares.
- `periodo_com_chuva` e `periodo_com_neve` resumem condições climáticas que podem reduzir a demanda.
- `horario_pico` marca períodos típicos de deslocamento urbano.
- `hora` é mantida porque é útil para a linha de base temporal, para modelos baseados em árvores e para a análise de erro por hora.

Não foi aplicada seleção automática de atributos, como `SelectKBest`, eliminação recursiva ou seleção por Lasso. Essa decisão foi tomada porque o conjunto de variáveis é pequeno, interpretável e já foi controlado manualmente para evitar vazamento de dados; para o escopo deste projeto, a adição de uma etapa automática poderia dificultar a interpretação tendo em vista a facilidade de avaliar as variáveis presentes.

Também não foram removidos picos de demanda como outliers, pois valores extremos de aluguel podem representar eventos operacionais reais, como horários de pico, dias com clima favorável ou mudanças legítimas no padrão de uso.

As variáveis categóricas serão tratadas por codificação one-hot dentro de um `ColumnTransformer`, sempre ajustado apenas nos dados de treinamento ou nos folds de validação cruzada.

In [ ]:
feature_columns = [
    "hora",
    "hora_seno",
    "hora_cosseno",
    "temperatura_c",
    "umidade_pct",
    "velocidade_vento_ms",
    "visibilidade_10m",
    "ponto_orvalho_c",
    "radiacao_solar_mj_m2",
    "chuva_mm",
    "neve_cm",
    "estacao",
    "feriado",
    "dia_funcionamento",
    "mes",
    "dia_semana",
    "fim_de_semana",
    "periodo_com_chuva",
    "periodo_com_neve",
    "horario_pico",
]

numeric_features = [
    "hora",
    "hora_seno",
    "hora_cosseno",
    "temperatura_c",
    "umidade_pct",
    "velocidade_vento_ms",
    "visibilidade_10m",
    "ponto_orvalho_c",
    "radiacao_solar_mj_m2",
    "chuva_mm",
    "neve_cm",
    "mes",
    "dia_semana",
]

categorical_features = [
    "estacao",
    "feriado",
    "dia_funcionamento",
    "fim_de_semana",
    "periodo_com_chuva",
    "periodo_com_neve",
    "horario_pico",
]

model_df = df[[DATE_COLUMN, TARGET] + feature_columns].copy()
print(f"Total de variáveis explicativas: {len(feature_columns)}")
model_df.head()

# 7. Divisão temporal dos dados

A divisão será cronológica: os primeiros 70% dos registros formam o treinamento, os 15% seguintes formam a validação e os 15% finais formam o teste. Isso simula um cenário realista em que o modelo é treinado no passado e avaliado em períodos futuros.

O conjunto de teste não será usado para escolher modelos nem hiperparâmetros.

In [ ]:
def temporal_train_validation_test_split(
    data: pd.DataFrame,
    train_size: float = TRAIN_SIZE,
    validation_size: float = VALIDATION_SIZE,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Divide uma base ordenada no tempo em treino, validação e teste."""
    n_rows = len(data)
    train_end = int(n_rows * train_size)
    validation_end = int(n_rows * (train_size + validation_size))
    train_data = data.iloc[:train_end].copy()
    validation_data = data.iloc[train_end:validation_end].copy()
    test_data = data.iloc[validation_end:].copy()
    return train_data, validation_data, test_data

train_df, validation_df, test_df = temporal_train_validation_test_split(model_df)

X_train = train_df[feature_columns]
y_train = train_df[TARGET]
X_validation = validation_df[feature_columns]
y_validation = validation_df[TARGET]
X_test = test_df[feature_columns]
y_test = test_df[TARGET]

if not (train_df[DATE_COLUMN].max() < validation_df[DATE_COLUMN].min() < test_df[DATE_COLUMN].min()):
    raise ValueError("A divisão temporal não está estritamente ordenada.")

split_summary = pd.DataFrame({
    "partição": ["treino", "validação", "teste"],
    "linhas": [len(train_df), len(validation_df), len(test_df)],
    "percentual": [len(train_df) / len(model_df), len(validation_df) / len(model_df), len(test_df) / len(model_df)],
    "início": [train_df[DATE_COLUMN].min(), validation_df[DATE_COLUMN].min(), test_df[DATE_COLUMN].min()],
    "fim": [train_df[DATE_COLUMN].max(), validation_df[DATE_COLUMN].max(), test_df[DATE_COLUMN].max()],
})

partitions = {
    "treino": train_df,
    "validação": validation_df,
    "teste": test_df,
}

split_composition = pd.concat(
    [
        partition_data.assign(partição=partition_name)[["partição", "estacao", TARGET]]
        for partition_name, partition_data in partitions.items()
    ],
    axis=0,
)

split_composition = (
    split_composition
    .groupby(["partição", "estacao"], as_index=False)
    .agg(linhas=(TARGET, "size"), demanda_média=(TARGET, "mean"))
    .sort_values(["partição", "estacao"])
)

print("Resumo cronológico da divisão:")
display(split_summary)
print("Composição por estação em cada partição:")
display(split_composition)

Esta separação tem o intuito de respeitar a ordem temporal e manter o teste como o período mais recente. Isso torna a avaliação mais realista do que uma divisão aleatória, mas também cria uma limitação: o conjunto de teste cobre apenas uma janela final do ano. Assim, análises por estação no teste devem ser lidas como diagnóstico daquele período específico, não como comparação completa entre todas as estações.

# 8. Pipeline de pré-processamento

O pré-processamento será encapsulado em um `ColumnTransformer`. Variáveis numéricas recebem imputação pela mediana e padronização com `StandardScaler`. Variáveis categóricas recebem imputação pelo valor mais frequente e codificação one-hot com `handle_unknown="ignore"`.

Esse desenho evita vazamento de dados porque o ajuste do pré-processamento acontece dentro dos pipelines, usando apenas o subconjunto apropriado em cada treinamento ou fold. A saída categórica é mantida densa para ser compatível com o experimento de PCA.

In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    """Cria um codificador compatível com versões recentes e anteriores do scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor() -> ColumnTransformer:
    """Cria o transformador de colunas usado pelos modelos supervisionados."""
    numeric_pipeline = Pipeline(steps=[
        ("imputação", SimpleImputer(strategy="median")),
        ("padronização", StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputação", SimpleImputer(strategy="most_frequent")),
        ("one_hot", make_one_hot_encoder()),
    ])
    return ColumnTransformer(
        transformers=[
            ("numéricas", numeric_pipeline, numeric_features),
            ("categóricas", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

preprocessor = make_preprocessor()
preprocessor.fit(X_train)
processed_train = preprocessor.transform(X_train)

try:
    processed_feature_names = preprocessor.get_feature_names_out()
except AttributeError:
    processed_feature_names = [f"variável_{index}" for index in range(processed_train.shape[1])]

print(f"Dimensões após o pré-processamento: {processed_train.shape[1]} variáveis.")

# 9. Análise de dimensionalidade

A análise de PCA será feita depois do pré-processamento ajustado apenas no conjunto de treino. O objetivo é verificar quanta variância é explicada pelos componentes principais e avaliar se a redução de dimensionalidade pode simplificar o modelo sem prejudicar a previsão.

In [ ]:
pca_full = PCA(random_state=SEED)
pca_full.fit(processed_train)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker="o")
axes[0].axhline(0.90, color="#E45756", linestyle="--", label="90% da variância")
axes[0].axhline(0.95, color="#72B7B2", linestyle="--", label="95% da variância")
axes[0].set_title("Variância explicada acumulada pelo PCA")
axes[0].set_xlabel("Número de componentes principais")
axes[0].set_ylabel("Variância explicada acumulada")
axes[0].legend()

pca_2d = PCA(n_components=2, random_state=SEED)
train_2d = pca_2d.fit_transform(processed_train)
scatter = axes[1].scatter(train_2d[:, 0], train_2d[:, 1], c=y_train, cmap="viridis", s=8, alpha=0.45)
axes[1].set_title("Projeção do treino nos dois primeiros componentes")
axes[1].set_xlabel("Componente principal 1")
axes[1].set_ylabel("Componente principal 2")
fig.colorbar(scatter, ax=axes[1], label="Bicicletas alugadas")

plt.tight_layout()
plt.show()

components_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)
components_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)
print(f"Componentes para explicar 90% da variância: {components_90}")
print(f"Componentes para explicar 95% da variância: {components_95}")

A curva de variância acumulada mostra quantos componentes são necessários para preservar a maior parte da informação após o pré-processamento. O PCA é usado aqui como ferramenta de análise e como experimento com Ridge, não como escolha automática para o modelo final. Em dados tabulares, reduzir dimensionalidade pode simplificar o espaço de atributos, mas também pode prejudicar a previsão ao misturar variáveis interpretáveis ou descartar sinais úteis.

# 10. Definição das métricas

Serão usadas três métricas:

- MAE: erro médio absoluto em número de bicicletas, usado como métrica principal.
- RMSE: raiz do erro quadrático médio, mais sensível a erros grandes.
- R²: proporção de variância explicada pelo modelo, útil como visão geral de ajuste.

In [ ]:
def compute_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    """Calcula as métricas de regressão usadas no projeto."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def evaluate_model(
    name: str,
    model,
    X_train_data: pd.DataFrame,
    y_train_data: pd.Series,
    X_val_data: pd.DataFrame,
    y_val_data: pd.Series,
) -> dict[str, float | str]:
    """Avalia um modelo em treino e validação, retornando uma linha comparável."""
    train_predictions = model.predict(X_train_data)
    validation_predictions = model.predict(X_val_data)
    train_metrics = compute_metrics(y_train_data, train_predictions)
    validation_metrics = compute_metrics(y_val_data, validation_predictions)
    return {
        "modelo": name,
        "mae_treino": train_metrics["MAE"],
        "mae_validacao": validation_metrics["MAE"],
        "rmse_treino": train_metrics["RMSE"],
        "rmse_validacao": validation_metrics["RMSE"],
        "r2_treino": train_metrics["R2"],
        "r2_validacao": validation_metrics["R2"],
        "diferenca_mae_validacao_treino": validation_metrics["MAE"] - train_metrics["MAE"],
    }


def evaluate_single_split(name: str, model, X_data: pd.DataFrame, y_data: pd.Series, split_name: str) -> dict[str, float | str]:
    """Avalia um modelo em uma única partição."""
    predictions = model.predict(X_data)
    metrics = compute_metrics(y_data, predictions)
    return {"modelo": name, "partição": split_name, **metrics}


def plot_model_comparison(results_df: pd.DataFrame) -> None:
    """Plota MAE de treino/validação e RMSE de validação por modelo."""
    sorted_results = results_df.sort_values("mae_validacao")
    positions = np.arange(len(sorted_results))
    width = 0.38

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].bar(positions - width / 2, sorted_results["mae_treino"], width=width, label="treino", color="#4C78A8")
    axes[0].bar(positions + width / 2, sorted_results["mae_validacao"], width=width, label="validação", color="#F58518")
    axes[0].set_title("MAE de treino e validação por modelo")
    axes[0].set_xlabel("Modelo")
    axes[0].set_ylabel("MAE em bicicletas por hora")
    axes[0].set_xticks(positions)
    axes[0].set_xticklabels(sorted_results["modelo"], rotation=45, ha="right")
    axes[0].legend()

    axes[1].barh(sorted_results["modelo"], sorted_results["rmse_validacao"], color="#54A24B")
    axes[1].invert_yaxis()
    axes[1].set_title("RMSE de validação por modelo")
    axes[1].set_xlabel("RMSE em bicicletas por hora")
    axes[1].set_ylabel("Modelo")

    plt.tight_layout()
    plt.show()

print("Funções de métricas definidas.")

# 11. Experimentos de modelagem

Os experimentos seguem uma progressão incremental:

- E0: linha de base ingênua com a média do treino.
- E1: linha de base de domínio com média por hora e estação.
- E2: regressão Ridge com pré-processamento.
- E3: PCA seguido de regressão Ridge.
- E4: Random Forest.
- E5: HistGradientBoosting.

A comparação inicial usa treino e validação. O teste permanece reservado.

In [ ]:
class HourSeasonAverageRegressor(BaseEstimator, RegressorMixin):
    """Linha de base que prevê a média histórica por hora e estação."""

    def __init__(self, hour_column: str = "hora", season_column: str = "estacao"):
        self.hour_column = hour_column
        self.season_column = season_column

    def fit(self, X: pd.DataFrame, y: pd.Series):
        training_data = X[[self.hour_column, self.season_column]].copy()
        training_data["alvo"] = np.asarray(y)
        self.global_mean_ = float(training_data["alvo"].mean())
        self.hour_means_ = training_data.groupby(self.hour_column)["alvo"].mean().to_dict()
        self.group_means_ = training_data.groupby([self.hour_column, self.season_column])["alvo"].mean().to_dict()
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        predictions = []
        for _, row in X[[self.hour_column, self.season_column]].iterrows():
            group_key = (row[self.hour_column], row[self.season_column])
            hour_key = row[self.hour_column]
            prediction = self.group_means_.get(group_key, self.hour_means_.get(hour_key, self.global_mean_))
            predictions.append(prediction)
        return np.asarray(predictions)


def make_model_candidates() -> dict[str, object]:
    """Cria os modelos candidatos com objetos novos a cada execução."""
    return {
        "E0 - média do treino": Pipeline(steps=[
            ("preprocessamento", make_preprocessor()),
            ("modelo", DummyRegressor(strategy="mean")),
        ]),
        "E1 - média por hora e estação": HourSeasonAverageRegressor(),
        "E2 - Ridge": Pipeline(steps=[
            ("preprocessamento", make_preprocessor()),
            ("modelo", Ridge(alpha=1.0)),
        ]),
        "E3 - PCA mais Ridge": Pipeline(steps=[
            ("preprocessamento", make_preprocessor()),
            ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
            ("modelo", Ridge(alpha=1.0)),
        ]),
        "E4 - Random Forest": Pipeline(steps=[
            ("preprocessamento", make_preprocessor()),
            ("modelo", RandomForestRegressor(
                n_estimators=250,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "E5 - HistGradientBoosting": Pipeline(steps=[
            ("preprocessamento", make_preprocessor()),
            ("modelo", HistGradientBoostingRegressor(
                max_iter=300,
                learning_rate=0.06,
                max_leaf_nodes=31,
                l2_regularization=0.0,
                early_stopping=False,
                random_state=RANDOM_STATE,
            )),
        ]),
    }

fitted_models = {}
metric_rows = []

for model_name, model in make_model_candidates().items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    fitted_models[model_name] = fitted_model
    metric_rows.append(evaluate_model(model_name, fitted_model, X_train, y_train, X_validation, y_validation))

initial_metrics = pd.DataFrame(metric_rows).sort_values("mae_validacao").reset_index(drop=True)
initial_metrics

# 12. Comparação inicial dos modelos

Nesta etapa, iremos considerar principalmente o MAE de validação. A diferença entre MAE de treino e validação ajudará a identificar sinais de underfitting ou de overfitting.

In [ ]:
best_initial_model_name = initial_metrics.loc[0, "modelo"]
print(f"Melhor modelo inicial por MAE de validação: {best_initial_model_name}")

display(initial_metrics)
plot_model_comparison(initial_metrics)

In [ ]:
def get_validation_mae(model_name: str) -> float:
    """Obtém o MAE de validação de um modelo pelo nome."""
    row = initial_metrics[initial_metrics["modelo"] == model_name]
    return float(row["mae_validacao"].iloc[0])

ridge_mae = get_validation_mae("E2 - Ridge")
pca_ridge_mae = get_validation_mae("E3 - PCA mais Ridge")
linear_best_mae = min(ridge_mae, pca_ridge_mae)
nonlinear_best_mae = initial_metrics[
    initial_metrics["modelo"].str.contains("Random Forest|HistGradientBoosting")
]["mae_validacao"].min()
large_gap_models = initial_metrics[initial_metrics["diferenca_mae_validacao_treino"] > 150]["modelo"].tolist()
baseline_mae = get_validation_mae("E0 - média do treino")
best_mae = float(initial_metrics.loc[0, "mae_validacao"])

print(f"Melhor modelo inicial por MAE de validação: {best_initial_model_name}.")
print(f"Ganho de MAE contra a linha de base ingênua: {baseline_mae - best_mae:.1f} bicicletas por hora.")

if pca_ridge_mae < ridge_mae:
    print("Nesta execução, o PCA reduziu o MAE de validação da regressão Ridge.")
elif pca_ridge_mae > ridge_mae:
    print("Nesta execução, o PCA aumentou o MAE de validação da regressão Ridge, sugerindo perda de informação preditiva relevante.")
else:
    print("Nesta execução, o PCA não alterou o MAE de validação da regressão Ridge.")

if nonlinear_best_mae < linear_best_mae:
    print("Os modelos não lineares melhoraram o MAE de validação em relação aos modelos lineares avaliados.")
else:
    print("Os modelos não lineares não superaram os modelos lineares nesta validação inicial.")

if large_gap_models:
    print("Modelos com maior sinal de overfitting pelo gap de MAE:", ", ".join(large_gap_models))
else:
    print("Nenhum modelo apresentou gap de MAE muito elevado pelo critério simples adotado.")

if best_mae > 0.8 * baseline_mae:
    print("Há sinal de underfitting geral, pois o melhor modelo ainda fica próximo da linha de base ingênua.")
else:
    print("Não há sinal forte de underfitting geral, pois o melhor modelo melhora substancialmente a linha de base ingênua.")

A célula anterior identifica o melhor modelo durante a validação automaticamente. Quando Random Forest ou HistGradientBoosting superam Ridge, há evidência de que relações não lineares e interações entre hora, clima, estação e operação são relevantes. A comparação entre Ridge e PCA mais Ridge mostra se a redução de dimensionalidade preservou ou descartou informação preditiva. Diferenças grandes entre MAE de treino e validação sugerem overfitting; em caso de desempenho próximo ao `DummyRegressor`, isso sugeriria underfitting.

# 13. Otimização de hiperparâmetros

A otimização será feita com `TimeSeriesSplit`, que preserva a ordem temporal dentro da validação cruzada. O modelo escolhido para ajuste é o `HistGradientBoostingRegressor`, porque ele combina bom desempenho inicial, capacidade de capturar não linearidades e custo computacional adequado.

Os hiperparâmetros ajustados controlam taxa de aprendizado, número de iterações, número máximo de folhas, tamanho mínimo de folha e regularização L2. O espaço de busca é intencionalmente pequeno para manter a execução confortável, reprodutível e adequada ao escopo deste MVP.

Para evitar vazamento de validação, a busca é ajustada apenas no conjunto de treino, usando validação cruzada temporal interna. Depois disso, o melhor conjunto de hiperparâmetros é comparado com a versão não otimizada no conjunto de validação reservado. Somente após essa escolha o modelo final é reajustado com treino e validação combinados. O teste continua reservado para a avaliação final.

Como o MAE foi definido como métrica principal por ser diretamente interpretável em bicicletas por hora, a versão não otimizada do `HistGradientBoostingRegressor` foi mantida como escolha final quando apresentou menor MAE de validação, ainda que a versão otimizada tenha melhorado RMSE e R².

In [ ]:
hgb_pipeline = Pipeline(steps=[
    ("preprocessamento", make_preprocessor()),
    ("modelo", HistGradientBoostingRegressor(
        early_stopping=False,
        random_state=RANDOM_STATE,
    )),
])

parameter_grid = {
    "modelo__learning_rate": [0.03, 0.05, 0.08, 0.12],
    "modelo__max_iter": [200, 300, 450],
    "modelo__max_leaf_nodes": [15, 31, 63],
    "modelo__min_samples_leaf": [15, 25, 40],
    "modelo__l2_regularization": [0.0, 0.01, 0.1, 1.0],
}

time_series_cv = TimeSeriesSplit(n_splits=4)

search = RandomizedSearchCV(
    estimator=hgb_pipeline,
    param_distributions=parameter_grid,
    n_iter=12,
    scoring="neg_mean_absolute_error",
    cv=time_series_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
    refit=True,
)

search.fit(X_train, y_train)

X_train_validation = pd.concat([X_train, X_validation], axis=0)
y_train_validation = pd.concat([y_train, y_validation], axis=0)

tuning_results = pd.DataFrame(search.cv_results_)
tuning_summary = (
    tuning_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]]
    .assign(
        média_mae_validação_cruzada=lambda data: -data["mean_test_score"],
        desvio_mae_validação_cruzada=lambda data: data["std_test_score"],
    )
    .sort_values("rank_test_score")
    .drop(columns=["mean_test_score", "std_test_score"])
    .rename(columns={"rank_test_score": "posição", "params": "parâmetros"})
    .reset_index(drop=True)
)

best_params_table = pd.DataFrame({
    "hiperparâmetro": list(search.best_params_.keys()),
    "valor": list(search.best_params_.values()),
})

untuned_hgb_name = "E5 - HistGradientBoosting sem otimização"
tuned_hgb_validation_name = "E6 - HistGradientBoosting otimizado"
untuned_hgb_validation_row = evaluate_model(
    untuned_hgb_name,
    fitted_models["E5 - HistGradientBoosting"],
    X_train,
    y_train,
    X_validation,
    y_validation,
)
tuned_hgb_validation_row = evaluate_model(
    tuned_hgb_validation_name,
    search.best_estimator_,
    X_train,
    y_train,
    X_validation,
    y_validation,
)

tuning_validation_comparison = (
    pd.DataFrame([untuned_hgb_validation_row, tuned_hgb_validation_row])
    .sort_values("mae_validacao")
    .reset_index(drop=True)
)

selected_hgb_model_name = str(tuning_validation_comparison.loc[0, "modelo"])
selected_hgb_validation_mae = float(tuning_validation_comparison.loc[0, "mae_validacao"])

if selected_hgb_model_name == tuned_hgb_validation_name:
    selected_hgb_model_for_refit = clone(search.best_estimator_)
else:
    selected_hgb_model_for_refit = clone(fitted_models["E5 - HistGradientBoosting"])

initial_hgb_validation_mae = untuned_hgb_validation_row["mae_validacao"]
tuned_hgb_validation_mae = tuned_hgb_validation_row["mae_validacao"]
best_cv_mae = -search.best_score_

print("Melhores hiperparâmetros encontrados apenas com o conjunto de treino:")
display(best_params_table)
print(f"Melhor MAE médio na validação cruzada temporal interna: {best_cv_mae:.1f}")
print(f"MAE de validação do HistGradientBoosting sem otimização: {initial_hgb_validation_mae:.1f}")
print(f"MAE de validação do HistGradientBoosting otimizado: {tuned_hgb_validation_mae:.1f}")
print("Comparação direta no conjunto de validação reservado:")
display(tuning_validation_comparison)
print(f"Configuração escolhida para o modelo final: {selected_hgb_model_name}, com MAE de validação {selected_hgb_validation_mae:.1f}.")
print("Melhores configurações segundo a validação cruzada temporal:")
tuning_summary.head(10)

# 14. Avaliação final no conjunto de teste

Após a seleção do modelo, a avaliação final é feita uma única vez no conjunto de teste, que representa o período mais recente e não foi usado para seleção de modelos ou hiperparâmetros.

O modelo final usa a configuração do `HistGradientBoostingRegressor` com menor MAE no conjunto de validação reservado, seja ela a versão otimizada ou a versão sem otimização. Em seguida, essa configuração é reajustada com treino e validação combinados. Esse reajuste é aceitável porque a escolha do modelo já foi concluída; o conjunto de teste continua intocado até a célula abaixo.

In [ ]:
final_model = clone(selected_hgb_model_for_refit)
final_model.fit(X_train_validation, y_train_validation)
final_model_name = f"Modelo final - {selected_hgb_model_name}"

final_metric_rows = [
    evaluate_single_split(final_model_name, final_model, X_train_validation, y_train_validation, "treino e validação"),
    evaluate_single_split(final_model_name, final_model, X_test, y_test, "teste"),
]

final_metrics = pd.DataFrame(final_metric_rows)
test_predictions = final_model.predict(X_test)

test_results = test_df[[DATE_COLUMN, TARGET, "hora", "estacao", "periodo_com_chuva"]].copy()
test_results["previsão"] = test_predictions
test_results["resíduo"] = test_results[TARGET] - test_results["previsão"]
test_results["erro_absoluto"] = test_results["resíduo"].abs()

final_metrics

# 15. Análise de erros

A análise de erros examina onde o modelo erra mais: na relação entre valores reais e previstos, na distribuição dos resíduos, em diferentes horas do dia, em estações do ano e em períodos com ou sem chuva.

Essa etapa ajuda a transformar métricas agregadas em diagnóstico prático.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(test_results[TARGET], test_results["previsão"], alpha=0.45, s=12, color="#4C78A8")
max_value = max(test_results[TARGET].max(), test_results["previsão"].max())
axes[0, 0].plot([0, max_value], [0, max_value], color="#E45756", linestyle="--")
axes[0, 0].set_title("Demanda real versus prevista")
axes[0, 0].set_xlabel("Demanda real")
axes[0, 0].set_ylabel("Demanda prevista")

axes[0, 1].hist(test_results["resíduo"], bins=35, color="#F58518", edgecolor="white")
axes[0, 1].axvline(0, color="black", linestyle="--")
axes[0, 1].set_title("Distribuição dos resíduos")
axes[0, 1].set_xlabel("Resíduo: real menos previsto")
axes[0, 1].set_ylabel("Frequência")

error_by_hour = test_results.groupby("hora", as_index=False)["erro_absoluto"].mean()
axes[1, 0].plot(error_by_hour["hora"], error_by_hour["erro_absoluto"], marker="o", color="#54A24B")
axes[1, 0].set_title("Erro absoluto médio por hora")
axes[1, 0].set_xlabel("Hora do dia")
axes[1, 0].set_ylabel("Erro absoluto médio")
axes[1, 0].set_xticks(range(0, 24, 2))

sample_days = 14 * 24
axes[1, 1].plot(test_results[DATE_COLUMN].iloc[:sample_days], test_results[TARGET].iloc[:sample_days], label="real", color="#4C78A8")
axes[1, 1].plot(test_results[DATE_COLUMN].iloc[:sample_days], test_results["previsão"].iloc[:sample_days], label="previsto", color="#E45756", alpha=0.85)
axes[1, 1].set_title("Série real e prevista nos primeiros 14 dias do teste")
axes[1, 1].set_xlabel("Data e hora")
axes[1, 1].set_ylabel("Bicicletas alugadas")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
error_by_season = (
    test_results.groupby("estacao", as_index=False)
    .agg(linhas=("erro_absoluto", "size"), erro_absoluto_médio=("erro_absoluto", "mean"))
    .sort_values("erro_absoluto_médio", ascending=False)
)

error_by_rain = (
    test_results.groupby("periodo_com_chuva", as_index=False)
    .agg(linhas=("erro_absoluto", "size"), erro_absoluto_médio=("erro_absoluto", "mean"))
    .sort_values("erro_absoluto_médio", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(error_by_season["estacao"], error_by_season["erro_absoluto_médio"], color="#72B7B2")
axes[0].set_title("Erro absoluto médio por estação no teste")
axes[0].set_xlabel("Estação")
axes[0].set_ylabel("Erro absoluto médio")

axes[1].bar(error_by_rain["periodo_com_chuva"], error_by_rain["erro_absoluto_médio"], color="#B279A2")
axes[1].set_title("Erro absoluto médio em períodos com e sem chuva")
axes[1].set_xlabel("Período com chuva")
axes[1].set_ylabel("Erro absoluto médio")

plt.tight_layout()
plt.show()

print("Erro por estação no teste; observe a quantidade de linhas antes de comparar grupos.")
display(error_by_season)
print("Erro por chuva no teste.")
display(error_by_rain)

O gráfico temporal mostra se o modelo acompanha bem o nível geral e a sazonalidade diária do período de teste. Diferenças maiores costumam aparecer em picos de demanda, quando pequenas mudanças de clima, rotina ou operação podem gerar aumentos rápidos de aluguel. Se os resíduos ficam mais negativos nos picos, o modelo está subestimando alta demanda; se ficam mais positivos, está superestimando períodos de baixa. A comparação por chuva ajuda a verificar se condições climáticas adversas aumentam o erro, o que teria implicação operacional direta: em dias chuvosos ou instáveis, a operação deveria tratar as previsões com maior margem de segurança.

A análise por estação no teste deve ser interpretada com cautela porque a divisão cronológica deixou o teste concentrado no período final da série. Portanto, ela descreve o comportamento nesse recorte temporal, mas não permite concluir que o modelo teria o mesmo erro relativo em todas as estações do ano.

# 16. Interpretabilidade

Como o modelo final é não linear, uma forma simples e independente do algoritmo para analisar importância é a importância por permutação. A ideia é embaralhar uma variável por vez e medir quanto o desempenho piora.

A interpretação deve ser feita com cautela: variáveis correlacionadas podem dividir importância, e a análise mostra associação preditiva, não causalidade.

In [ ]:
permutation = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=8,
    random_state=SEED,
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "variável": feature_columns,
        "aumento_médio_mae": permutation.importances_mean,
        "desvio_padrão": permutation.importances_std,
    })
    .sort_values("aumento_médio_mae", ascending=False)
    .reset_index(drop=True)
)

top_importance = importance_table.head(12).sort_values("aumento_médio_mae")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_importance["variável"], top_importance["aumento_médio_mae"], color="#4C78A8")
ax.set_title("Principais variáveis por importância de permutação")
ax.set_xlabel("Aumento médio do MAE ao embaralhar a variável")
ax.set_ylabel("Variável")
plt.tight_layout()
plt.show()

importance_table.head(12)

# 17. Discussão

A divisão temporal reduz o risco de avaliar o modelo em um cenário artificialmente fácil, enquanto os pipelines garantem que o pré-processamento seja ajustado apenas nos dados disponíveis em cada etapa.

A linha de base ingênua mediu o ganho mínimo esperado em relação à média histórica. A linha de base por hora e estação adiciona conhecimento de domínio simples e costuma ser uma referência mais justa para séries com forte sazonalidade diária. Os modelos Ridge testaram uma relação linear regularizada, enquanto Random Forest e HistGradientBoosting capturaram interações e não linearidades entre clima, horário, estação e funcionamento do serviço.

A análise de PCA verificou se uma representação compacta ajuda o modelo linear. Em problemas tabulares com poucas variáveis e categorias informativas, PCA pode reduzir ruído, mas também pode descartar sinais úteis e dificultar interpretação. Por isso, a decisão deve ser guiada pelo MAE de validação, não apenas pela variância explicada.

O tempo de treinamento e os recursos computacionais não foram um gargalo relevante neste MVP. A base tem apenas 8.760 registros horários e os modelos utilizados pertencem ao ecossistema `scikit-learn`, executando confortavelmente em CPU em ambiente de notebook/Colab. Mesmo assim, o espaço de busca de hiperparâmetros foi mantido pequeno para preservar reprodutibilidade, reduzir tempo de execução e evitar que a complexidade computacional se tornasse desproporcional ao objetivo acadêmico do projeto.

Principais limitações:

- A base cobre apenas uma cidade e um período histórico específico.
- Não foram usadas variáveis externas, como eventos, greves, preço, disponibilidade de bicicletas ou dados de estações individuais.
- O modelo prevê demanda agregada por hora, não demanda por estação de retirada.
- A variável `dia_funcionamento` é tratada como informação operacional conhecida antes da previsão; se isso não estiver disponível em outro contexto, ela não deve ser usada da mesma forma.
- O conjunto de teste representa o período final da série e não cobre todas as estações, o que limita comparações sazonais na avaliação final.
- Não foram criadas variáveis defasadas ou médias móveis para evitar complexidade adicional neste MVP.
- A importância por permutação indica relevância preditiva, mas não prova causalidade.

Melhorias futuras:

- Criar variáveis defasadas usando apenas informações passadas.
- Testar validação temporal com janelas deslizantes.
- Incluir dados externos de calendário, eventos e transporte público.
- Melhorar a aplicação operacional do modelo incluindo médias móveis de demanda, disponibilidade por estação, eventos locais e previsões meteorológicas mais ricas, especialmente para reduzir erros em horários de pico que afetam o reposicionamento de bicicletas.
- Modelar incerteza das previsões com intervalos preditivos.
- Comparar desempenho por mês, estação e condições climáticas extremas com janelas de teste mais representativas.
- Avaliar estratégias específicas para períodos de chuva, neve e baixa operação.

In [ ]:
test_mae = float(final_metrics.loc[final_metrics["partição"] == "teste", "MAE"].iloc[0])
test_rmse = float(final_metrics.loc[final_metrics["partição"] == "teste", "RMSE"].iloc[0])
test_r2 = float(final_metrics.loc[final_metrics["partição"] == "teste", "R2"].iloc[0])
most_important_feature = str(importance_table.loc[0, "variável"])

print(f"No conjunto de teste, o modelo final apresentou MAE de {test_mae:.1f} bicicletas por hora.")
print(f"O RMSE foi {test_rmse:.1f} e o R² foi {test_r2:.3f}.")
print(f"A variável mais relevante na análise de permutação foi: {most_important_feature}.")
print("Esses números devem ser lidos em conjunto com os gráficos de erro, pois médias agregadas podem esconder períodos críticos.")

# 18. Conclusão

O notebook constrói uma solução reprodutível para previsão de demanda horária de bicicletas compartilhadas em Seul. A base pública da UCI contém registros horários de demanda, clima, calendário, estação do ano, feriados e funcionamento do serviço. O alvo é `bicicletas_alugadas`, tratado como problema de regressão supervisionada com estrutura temporal.

O pré-processamento foi encapsulado em `Pipeline` e `ColumnTransformer`, com imputação, padronização e one-hot encoding ajustados apenas nos dados apropriados de cada etapa. A abordagem começa com linhas de base simples, evolui para Ridge, PCA mais Ridge, Random Forest e HistGradientBoosting, ajusta hiperparâmetros com validação temporal e finaliza com avaliação em um conjunto de teste separado.

Do ponto de vista de desempenho, o modelo final apresentou MAE de 171,4 bicicletas por hora, RMSE de 245,5 e R² de 0,814 no conjunto de teste. Como o objetivo principal deste MVP não era maximizar performance a qualquer custo, esses números devem ser interpretados como evidência de que o fluxo de modelagem capturou sinal relevante nos dados, especialmente quando comparado às linhas de base simples. Ainda assim, a diferença entre o erro em treino/validação e o erro em teste, além dos erros maiores em horários de pico, indica que o modelo ainda não deve ser tratado como uma solução operacional pronta sem validações adicionais.

A análise de erros complementa essa visão ao mostrar onde o modelo erra mais por hora, chuva e recorte sazonal disponível no teste. A importância por permutação sugere que temperatura, funcionamento do serviço e hora do dia foram variáveis centrais para a previsão, o que é coerente com o comportamento esperado de um sistema de bicicletas compartilhadas.

As principais limitações são a ausência de dados externos, a granularidade agregada por hora, a dependência da disponibilidade prévia de `dia_funcionamento` e a cobertura sazonal restrita do conjunto de teste. Como próximos passos, o projeto poderia incluir defasagens seguras, janelas temporais deslizantes, dados externos e avaliação mais detalhada por estação e condições climáticas extremas.

O projeto atende ao objetivo principal: demonstrar um fluxo sólido de regressão supervisionada com estrutura temporal e cuidado contra vazamento de dados.